# 🐜🐝 Mini proiect · Transfer learning pe propriile imagini

În acest notebook vei adapta un model pre-antrenat la **propriile tale imagini**. Poți clasifica animale, obiecte, plante sau orice altceva pentru care ai cel puțin două clase de imagini.

Vei folosi ResNet18, un model care a învățat deja să recunoască forme și texturi. Tu îi vei schimba ultimul strat ca să învețe clasele proiectului tău.

## Cum rulezi în Google Colab

1. Deschide notebook-ul în Google Colab. Alege `Runtime` → `Change runtime type` → `T4 GPU` → `Save`.
2. Pregătește imaginile în structura de mai jos. Numele folderelor sunt numele claselor:

```text
/content/proiectul_meu/
├── train/
│   ├── clasa_1/   # aproximativ 80% dintre imagini
│   └── clasa_2/
└── val/
    ├── clasa_1/   # aproximativ 20% dintre imagini
    └── clasa_2/
```

3. Nu pune aceeași imagine în `train` și `val`. Ideal, începe cu 50–100 de imagini pentru fiecare clasă.
4. În Colab, deschide pictograma de folder din stânga și încarcă datele în `/content/`. Dacă folosești Google Drive, montează Drive și folosește o cale care începe cu `/content/drive/MyDrive/`.
5. Completează doar setările din celula marcată **SETĂRI PROIECT**, apoi rulează celulele în ordine.

In [8]:
# Importăm doar bibliotecile necesare pentru acest mini-proiect.
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms, models

# Folosim GPU dacă este disponibil în Colab; altfel folosim procesorul.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Rulăm pe:", device)

Rulăm pe: cuda


In [9]:

from pathlib import Path
from sklearn.model_selection import train_test_split
import shutil

source = Path("/content/plants")
destination = Path("/content/proiectul_meu")

classes = [
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy"
]

for cls in classes:
    # Folosim folderele reale, nu cele din __MACOSX
    folder = source / cls

    images = [
        p for p in folder.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]

    train, val = train_test_split(
        images, test_size=0.2, random_state=42
    )

    for split, files in [("train", train), ("val", val)]:
        target = destination / split / cls
        target.mkdir(parents=True, exist_ok=True)

        for img in files:
            shutil.copy2(img, target / img.name)

    print(f"{cls}: {len(train)} train, {len(val)} val")

print("Dataset pregatit!")



Potato___Early_blight: 800 train, 200 val
Potato___Late_blight: 800 train, 200 val
Potato___healthy: 121 train, 31 val
Dataset pregatit!


## Funcții ajutătoare

Nu trebuie să modifici codul din celula următoare. Funcțiile se ocupă de antrenare și evaluare, ca `fit()` și `score()` în scikit-learn.

In [10]:
def evalueaza(model, loader):
    # Oprim calculul gradientului: acum doar verificăm modelul.
    model.eval()
    corecte = 0
    total = 0

    with torch.no_grad():
        for imagini, etichete in loader:
            imagini = imagini.to(device)
            etichete = etichete.to(device)
            predictii = model(imagini).argmax(1)
            corecte += (predictii == etichete).sum().item()
            total += len(etichete)

    return corecte / total


def antreneaza(model, train_loader, val_loader, epoci, rata_invatare):
    model.to(device)

    # Antrenăm doar parametrii care nu sunt înghețați.
    parametri_antrenabili = [
        parametru for parametru in model.parameters()
        if parametru.requires_grad
    ]
    optimizer = torch.optim.Adam(parametri_antrenabili, lr=rata_invatare)
    functie_pierdere = nn.CrossEntropyLoss()

    for epoca in range(1, epoci + 1):
        model.train()
        suma_pierdere = 0

        for imagini, etichete in train_loader:
            imagini = imagini.to(device)
            etichete = etichete.to(device)

            optimizer.zero_grad()
            predictii = model(imagini)
            pierdere = functie_pierdere(predictii, etichete)
            pierdere.backward()
            optimizer.step()
            suma_pierdere += pierdere.item()

        acuratete_val = evalueaza(model, val_loader)
        print(
            f"Epoca {epoca}/{epoci} | "
            f"pierdere: {suma_pierdere:.3f} | "
            f"validare: {acuratete_val:.1%}"
        )

    return model

## Pasul 1 · Setările proiectului

Schimbă doar valorile din zona de mai jos. La prima rulare, păstrează `resnet18` și `feature_extraction`.

In [11]:
# ======== SETĂRI PROIECT: schimbă doar aici ========
CALE_DATE = "/content/proiectul_meu"  # folderul care conține train/ și val/
ARHITECTURA = "resnet18"             # păstrează această valoare la prima rulare
STRATEGIE = "feature_extraction"     # apoi poți încerca "fine_tuning"
EPOCI = 3                              # câte treceri face modelul prin imaginile de train
# ===================================================

## Pasul 2 · Verifică datele

`ImageFolder` citește automat numele folderelor ca nume de clase. Dacă apare o eroare, verifică mai întâi calea și structura folderelor.

In [12]:
date_train = datasets.ImageFolder(f"{CALE_DATE}/train")
date_val = datasets.ImageFolder(f"{CALE_DATE}/val")

CLASE = date_train.classes
print("Clase găsite:", CLASE)
print("Imagini pentru antrenare:", len(date_train))
print("Imagini pentru validare:", len(date_val))

Clase găsite: ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']
Imagini pentru antrenare: 1721
Imagini pentru validare: 431


## Pasul 3 · Construiește modelul

Modelul este pre-antrenat pe ImageNet. Înlocuim ultimul strat cu unul care are exact câte ieșiri ai clase în proiect.

In [13]:
greutati = models.get_model_weights(ARHITECTURA).DEFAULT
model = models.get_model(ARHITECTURA, weights=greutati)

# Feature extraction: păstrăm straturile învățate și antrenăm doar capul.
# Fine-tuning: permitem și straturilor modelului să se adapteze.
for parametru in model.parameters():
    parametru.requires_grad = (STRATEGIE == "fine_tuning")

# ResNet are ultimul strat în model.fc.
if hasattr(model, "fc"):
    model.fc = nn.Linear(model.fc.in_features, len(CLASE))
else:
    # EfficientNet și MobileNet au ultimul strat în model.classifier.
    model.classifier[-1] = nn.Linear(
        model.classifier[-1].in_features, len(CLASE)
    )

print(model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 162MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

## Pasul 4 · Pregătește imaginile

La antrenare folosim augmentare, pentru ca modelul să vadă imagini ușor diferite. La validare folosim doar transformarea standard a modelului.

In [14]:
transformare_train = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    greutati.transforms(),
])

train_dataset = datasets.ImageFolder(
    f"{CALE_DATE}/train", transform=transformare_train
)
val_dataset = datasets.ImageFolder(
    f"{CALE_DATE}/val", transform=greutati.transforms()
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=16, shuffle=True
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=32
)
print("Datele sunt pregătite.")

Datele sunt pregătite.


## Pasul 5 · Antrenează și evaluează

Dacă ai puține imagini, începe cu `feature_extraction`. După aceea schimbă `STRATEGIE` în `fine_tuning` și compară rezultatele.

In [15]:
rata_invatare = 1e-3 if STRATEGIE == "feature_extraction" else 1e-4

model = antreneaza(
    model,
    train_loader,
    val_loader,
    epoci=EPOCI,
    rata_invatare=rata_invatare,
)

acuratete = evalueaza(model, val_loader)
print(f"Acuratețea finală pe validare: {acuratete:.1%}")

Epoca 1/3 | pierdere: 54.775 | validare: 88.6%
Epoca 2/3 | pierdere: 30.551 | validare: 85.8%
Epoca 3/3 | pierdere: 26.388 | validare: 89.8%
Acuratețea finală pe validare: 89.8%


## Provocare opțională

Schimbă `ARHITECTURA` în `"efficientnet_b0"` sau `"mobilenet_v3_large"` și rulează din nou de la Pasul 3. Compară acuratețea și timpul de antrenare cu ResNet18.

> ✍️ Ce arhitectură ai alege pentru proiectul tău și de ce?

## Recapitulare

- Datele trebuie organizate în `train/` și `val/`, cu un folder pentru fiecare clasă.
- `feature_extraction` antrenează doar ultimul strat și este un punct bun de pornire pentru seturi mici.
- `fine_tuning` permite și straturilor modelului să se adapteze, folosind o rată de învățare mai mică.
- Acuratețea este doar începutul: verifică și imaginile clasificate greșit înainte să alegi modelul final.